This notebook generates all plots regarding the ABC-SMC model selection and parameter inference, either model probabilites or prior/posterior distributions of the parameters.

In [34]:
import pyabc
from sympy import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import use
use('Agg')
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14) 

In [2]:
# Update the dataset of the binding rates
def update_df_bind_ABC(df_bind, param_dict, param_depen, param_unique_var, variants_list):
    # Create copy of binding dataframe
    df_bind_fit = df_bind.copy()

    # Update df_bind with new parameter values
    i=0
    for variant in variants_list:
        if variant == "WT":
            for param in [key.split('__')[1] for key in param_dict.keys() if key.split('__')[0] == variant]:
                # Copy new parameter to dataframe
                df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant).loc[pd.DataFrame(df_bind_fit["Variant"] == variant).Variant, :].index[0], param] = 10**param_dict[variant+"__"+param]
                # Parameter dependancies
                if param in param_depen.keys():
                    for param_d in param_depen[param]:
                        param1_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param]
                        param2_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param_d]
                        df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant).loc[pd.DataFrame(df_bind_fit["Variant"] == variant).Variant, :].index[0], param_d] = (10**param_dict[variant+"__"+param])/param1_0*param2_0
                # Parameter which are the same for WT and other variants must be also changed in the variant parameter row
                for variant_WT_mut in variants_list[1:]: # For all the non-WT variants
                    if param not in param_unique_var[variant_WT_mut]:
                        param_dep_bool = True # If parameter is not unique to the variant, then it must be changed following the WT
                        for param2 in param_unique_var[variant_WT_mut]:
                            if param2 in param_depen.keys(): 
                                if param == param_depen[param2]:
                                    param_dep_bool = False # If parameter is not unique to the variant, but is dependant to a unique one, then it must be changed following the variant parameter
                        # Change the variant non-unqiue parameter and the ones that depend on it, following the WT one
                        if param_dep_bool:
                            df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant_WT_mut).loc[pd.DataFrame(df_bind_fit["Variant"] == variant_WT_mut).Variant, :].index[0], param] = 10**param_dict[variant+"__"+param]
                            # Parameter dependancies
                            if param in param_depen.keys():
                                for param_d in param_depen[param]:
                                    param1_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param]
                                    param2_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param_d]
                                    df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant_WT_mut).loc[pd.DataFrame(df_bind_fit["Variant"] == variant_WT_mut).Variant, :].index[0], param_d] = (10**param_dict[variant+"__"+param])/param1_0*param2_0
        # If parameter is not specified as WT, then its unique to the variant -> change only parameter and the ones that depend on it
        else:
            for param in [key.split('__')[1] for key in param_dict.keys() if key.split('__')[0] == variant]:
                df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant).loc[pd.DataFrame(df_bind_fit["Variant"] == variant).Variant, :].index[0], param] = 10**param_dict[variant+"__"+param]
                # Parameter dependancies
                if param in param_depen.keys():
                    for param_d in param_depen[param]:
                        param1_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param]
                        param2_0 = df_bind.loc[pd.DataFrame(df_bind["Variant"] == variant).loc[pd.DataFrame(df_bind["Variant"] == variant).Variant, :].index[0], param_d]
                        df_bind_fit.loc[pd.DataFrame(df_bind_fit["Variant"] == variant).loc[pd.DataFrame(df_bind_fit["Variant"] == variant).Variant, :].index[0], param_d] = (10**param_dict[variant+"__"+param])/param1_0*param2_0
    return df_bind_fit

In [3]:
# Initialize datasets
directory_plots = "figures/ABC_SMC"
path_data = '..'
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10.csv') # Parameters to fit
df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]

# Get only data on certain variant
variants = ["WT","Super-10","R5A11D"]
df_bind = df_bind.loc[df_bind["Variant"].isin(variants)]

# Load model selection+parameter inference
history = pyabc.History("sqlite:///results/ABC_SMC/pyABC_model_selection.db")

### Plot model probabilities over the iterations

In [36]:
# Get model probabilities
df_ms = history.get_model_probabilities()
df_ms.columns = ["Control","IL10 RAp","IL10 MS1"]
# Generate figure model selection
fig,ax=plt.subplots(1,1,figsize=(12, 6), dpi=400)
plt.plot(df_ms.index,df_ms["Control"],color="darkblue",label="Control",linewidth=5,linestyle="dashed")
plt.plot(df_ms.index,df_ms["IL10 RAp"],color="darkred",label="Differential phosphorylation",linewidth=5)
plt.plot(df_ms.index,df_ms["IL10 MS1"],color="darkgreen",label="Kinetic proofreading",linewidth=5,linestyle="dotted")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
plt.ylabel('Probability', fontsize=20)
plt.xlabel('Iteration', fontsize=20)
ax.legend(loc="upper left", fontsize=14)
plt.savefig(directory_plots+'/model_selection.pdf', bbox_inches='tight', transparent=True)

In [35]:
# Get model probabilities
df_ms = history.get_model_probabilities()
df_ms.columns = ["Control","IL10 RAp","IL10 MS1"]

# Generate figure model selection
colors_bars = ['#669900','#8cbf26','#b2dc4c','#97cb73','#006699','#268cbf','#684a97','#a60d73','#cc3399','#f45b22','#ff8000','#ffa600','#ffcc00']
x = np.arange(len(df_ms.columns))  # the label locations
width = 0.07 # the width of the bars
multiplier = -5
fig,ax=plt.subplots(1,1,figsize=(12, 6), dpi=400)

i2 = 0
for i in np.arange(0,len(df_ms),2):
    offset = width * multiplier
    rects = ax.bar(x + offset, df_ms.loc[i].values, width, label="Iter. "+str(i),color=colors_bars[i2])
    multiplier += 1
    i2 += 1

ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel('Probability', fontsize=25)
ax.set_xticks(x + width, df_ms.columns)
ax.legend(loc="upper right", fontsize=12)
plt.savefig(directory_plots+'/model_selection.pdf', bbox_inches='tight', transparent=True)

### Plot and save parameter inference of WT + variants in model selection

In [23]:
directory_plots = "figures/ABC_SMC/"
history = pyabc.History("sqlite:///results/ABC_SMC/pyABC_model_selection.db")
conf95_2SD = 0.75
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
variants_list = ["WT","Super-10","R5A11D"]
model_list = ["Control","IL10_RAp"] # We don't include IL10_MS1 as it was not selected 
param_depen = {'k_IL_RB_f':['k_IL_RA_RB_f'],'k_IL_RA_RB_f':['k_IL_RB_f'],'k_IL_RB_b':['k_IL_RA_RB_b'],'k_IL_RA_RB_b':['k_IL_RB_b']}
param_unique_var = {"WT": df_bind.columns[1:],'R5A11D':['k_IL_RB_f','k_IL_RB_b','k_IL_RA_RB_f','k_IL_RA_RB_b'],'Super-10':['k_IL_RB_f','k_IL_RB_b','k_IL_RA_RB_f','k_IL_RA_RB_b']}
param_vec_plots_dict = {
    "Control": {'surf_cell':'$S_{cell}$','k_ILM_b':'$K_{ILM-b}$','k_IL_RA_b':'$K_{IL-RA-b}$','k_IL_RA_RB_b':'$K_{IL-RA-RB-b}$'},
    "IL10_RAp": {'surf_cell':'$S_{cell}$','k_ILM_b':'$K_{ILM-b}$','k_IL_RA_b':'$K_{IL-RA-b}$','k_IL_RA_RB_b':'$K_{IL-RA-RB-b}$','k_DEPHOS_R':'$K_{DEPHOS-R}$'}
    }
param_dict= {
    "Control": {},
    "IL10_RAp": {}
    }

### Save best individual of last iteration of IL10 RAp model

In [24]:
model_i = 1
model = model_list[1]
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
print("Prior/posterior distr. plots for: "+model)
fin_pop = [particle for particle in history.get_population(t=history.max_t).get_particles_by_model()[model_i]]
for variant in variants_list:
    df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10.csv') # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[(df_fit_data["Var__Param"].str.contains(variant))&(df_fit_data["Model"]==model)]
        
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_vec_plots_dict[model][param]

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        x = 10**x
        
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Get data from final time point
        df, w = history.get_distribution(m=model_i, t=history.max_t)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        
        # Change the binding rates dataset to include the fitted parameters (take the individual with lowest distance
        i = 0
        error_saved = 1
        i_saved = 0
        for particle in fin_pop:
            if particle.distance < error_saved:
                error_saved = particle.distance
                i_saved = i
            i += 1
        
        param_dict[model][var__param] = fin_pop[i_saved].parameter[var__param]
        
        # Get the evolution of the optimal parameter values with each iteration of the posterior
        for i in range(0,history.max_t+1):
            df, w = history.get_distribution(m=model_i, t=i)
            x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
            opt_param_evol.append(10**x[np.argmax(pdf)]) # Save value where prior is maximum for temporal evolution  
        
        # Save distribution channging from unbinding rate to Kd
        x = 10**x
                
        print("Optimal "+var+"_"+param+": " + str(10**fin_pop[i_saved].parameter[var__param]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Density", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        if model == "Control":
            plt.savefig(directory_plots+'IL10/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        else:
            plt.savefig(directory_plots+model+'/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(param_plot, fontsize=25)
        plt.yscale("log")
        plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
        middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
        plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        if model == "Control":
            plt.savefig(directory_plots +'IL10/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        else:
            plt.savefig(directory_plots +model+'/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    if model == "Control":
        plt.savefig(directory_plots +'IL10/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    else:
        plt.savefig(directory_plots +model+'/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.show()

Prior/posterior distr. plots for: IL10_RAp
Original WT_surf_cell: 1.3993553601551832e-08
Optimal WT_surf_cell: 2.486265780721942e-08
Original WT_k_ILM_b: 0.04338001616481063
Optimal WT_k_ILM_b: 0.42809162928994543
Original WT_k_IL_RA_b: 0.00021989869945295703
Optimal WT_k_IL_RA_b: 0.0014181974285853126
Original WT_k_IL_RA_RB_b: 1.2494244287099834
Optimal WT_k_IL_RA_RB_b: 0.18386605260832015
Original WT_k_DEPHOS_R: 0.9995395429679866
Optimal WT_k_DEPHOS_R: 0.31810505271543205
Original Super-10_k_IL_RA_RB_b: 0.013293875921474222
Optimal Super-10_k_IL_RA_RB_b: 0.03452944883632479
Original R5A11D_k_IL_RA_RB_b: 0.039981581718719475
Optimal R5A11D_k_IL_RA_RB_b: 0.1092168502212518


In [25]:
# Update binding rates to the infered values and save them
variants_list = ["WT","Super-10","R5A11D"]
df_bind = update_df_bind_ABC(df_bind, param_dict['IL10_RAp'], param_depen, param_unique_var, variants_list)
# Asssume proportionality of IL+RB and IL-RA+RB interaction as mutations are in the IL and RB interface
df_bind.loc[df_bind["Variant"]=="Super-10","k_IL_RB_b"] = df_bind.loc[df_bind["Variant"]=="Super-10","k_IL_RA_RB_b"].values[0]/df_bind.loc[df_bind["Variant"]=="R5A11D","k_IL_RA_RB_b"].values[0]*df_bind.loc[df_bind["Variant"]=="R5A11D","k_IL_RB_b"].values[0]
df_bind.to_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp.csv", index = False) # Save infered parameter values

### Save best individual of second-last iteration of IL10 control model (last iteration model died)

In [26]:
param_dict= {
    "Control": {},
    "IL10_RAp": {}
    }
variants_list = ["WT","Super-10","R5A11D"]
model_i = 0
model = model_list[0]
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
print("Prior/posterior distr. plots for: "+model)
fin_pop = [particle for particle in history.get_population(t=history.max_t-1).get_particles_by_model()[model_i]]
for variant in variants_list:
    df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10.csv') # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[(df_fit_data["Var__Param"].str.contains(variant))&(df_fit_data["Model"]==model)]
        
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_vec_plots_dict[model][param]

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        x = 10**x
        
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Get data from final time point
        df, w = history.get_distribution(m=model_i, t=history.max_t-1)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        
        # Change the binding rates dataset to include the fitted parameters (take the individual with lowest distance
        i = 0
        error_saved = 1
        i_saved = 0
        for particle in fin_pop:
            if particle.distance < error_saved:
                error_saved = particle.distance
                i_saved = i
            i += 1
        
        param_dict[model][var__param] = fin_pop[i_saved].parameter[var__param]
        
        # Get the evolution of the optimal parameter values with each iteration of the posterior
        for i in range(0,history.max_t-1+1):
            df, w = history.get_distribution(m=model_i, t=i)
            x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
            opt_param_evol.append(10**x[np.argmax(pdf)]) # Save value where prior is maximum for temporal evolution  
        
        # Save distribution channging from unbinding rate to Kd
        x = 10**x
                
        print("Optimal "+var+"_"+param+": " + str(10**fin_pop[i_saved].parameter[var__param]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Density", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        if model == "Control":
            plt.savefig(directory_plots+'IL10/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        else:
            plt.savefig(directory_plots+model+'/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t-1+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(param_plot, fontsize=25)
        plt.yscale("log")
        plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
        middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
        plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        if model == "Control":
            plt.savefig(directory_plots +'IL10/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        else:
            plt.savefig(directory_plots +model+'/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    if model == "Control":
        plt.savefig(directory_plots +'IL10/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    else:
        plt.savefig(directory_plots +model+'/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.show()

Prior/posterior distr. plots for: Control
Original WT_surf_cell: 1.3993553601551832e-08
Optimal WT_surf_cell: 5.533886679097193e-08
Original WT_k_ILM_b: 0.04338001616481063
Optimal WT_k_ILM_b: 0.12239823514155693
Original WT_k_IL_RA_b: 0.00021989869945295703
Optimal WT_k_IL_RA_b: 0.00022696544869925384
Original WT_k_IL_RA_RB_b: 1.2494244287099834
Optimal WT_k_IL_RA_RB_b: 0.16861820067296499
Original Super-10_k_IL_RA_RB_b: 0.013293875921474222
Optimal Super-10_k_IL_RA_RB_b: 0.07906149667978463
Original R5A11D_k_IL_RA_RB_b: 0.039981581718719475
Optimal R5A11D_k_IL_RA_RB_b: 0.10808865551371187


In [27]:
# Update binding rates to the infered values and save them
variants_list = ["WT","Super-10","R5A11D"]
df_bind = update_df_bind_ABC(df_bind, param_dict['Control'], param_depen, param_unique_var, variants_list)
df_bind.to_csv("data/binding/IL10_data_param_ABC_SMC_IL10_Control.csv", index = False) # Save infered parameter values

## Fine tuning using the full Receptor Memory model (numerically simulated ODE system)

### Save best individual and plot prior and posterior distributions

In [28]:
history = pyabc.History("sqlite:///results/ABC_SMC/pyABC_inference_ODE_WT.db")
param_dict= {
    "IL10_RAp": {}
    }
model_i = 0
model = model_list[1]
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
print("Prior/posterior distr. plots for: "+model)
fin_pop = [particle for particle in history.get_population(t=history.max_t).get_particles_by_model()[model_i]]
variant = "WT"
df_fit_data_ODE = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10_ODE_WT.csv') # Parameters to fit
df_fit_data_ODE["Var__Param"] = df_fit_data_ODE["Variant"]+"__"+df_fit_data_ODE["Parameter"]
df_fit_data_ODE = df_fit_data_ODE[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
df_fit_data_ODE = df_fit_data_ODE.loc[(df_fit_data_ODE["Var__Param"].str.contains(variant))&(df_fit_data_ODE["Model"]==model)]

df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10.csv') # Parameters to fit
df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
df_fit_data = df_fit_data.loc[(df_fit_data["Var__Param"].str.contains(variant))&(df_fit_data["Model"]==model)]
    
# For each infered parameter in the variant
for var__param in df_fit_data_ODE["Var__Param"]:
    var, param = var__param.split("__")
    param_plot = param_vec_plots_dict[model][param]

    # Plot prior and final posterior distributions
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    
    # Plot prior
    x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
    pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
    # If infered parameter is an unbinding rate in the plot we put the Kd
    x = 10**x
    
    opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
        
    # Plot prior
    plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
    
    print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
    
    # Get data from final time point
    df, w = history.get_distribution(m=model_i, t=history.max_t)
    x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
    
    # Change the binding rates dataset to include the fitted parameters (take the individual with lowest distance
    i = 0
    error_saved = 1
    i_saved = 0
    for particle in fin_pop:
        if particle.distance < error_saved:
            error_saved = particle.distance
            i_saved = i
        i += 1
    
    param_dict[model][var__param] = fin_pop[i_saved].parameter[var__param]
    
    # Get the evolution of the optimal parameter values with each iteration of the posterior
    for i in range(0,history.max_t+1):
        df, w = history.get_distribution(m=model_i, t=i)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        opt_param_evol.append(10**x[np.argmax(pdf)]) # Save value where prior is maximum for temporal evolution  
    
    # Save distribution channging from unbinding rate to Kd
    x = 10**x
            
    print("Optimal "+var+"_"+param+": " + str(10**fin_pop[i_saved].parameter[var__param]))    
    
    # Plot posterior
    plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

    # Finalize plot with prior and posterior distributions
    plt.ylabel("Density", fontsize=25)
    plt.xlabel(param_plot, fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.legend(loc="best", fontsize=20)
    ax.set_yticks([])
    plt.xscale('log')
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
    plt.show()
    
    # Plot evolution of optimal parameter values
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(param_plot, fontsize=25)
    plt.yscale("log")
    plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
    middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
    plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
    plt.show()
    
    
# Plot epsilon evolution
x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
plt.plot(x,y, color="blue", linewidth=7.5)
plt.xlabel("Iteration", fontsize=25)
plt.ylabel(r'$\epsilon$', fontsize=25)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
plt.show()

Prior/posterior distr. plots for: IL10_RAp
Original WT_surf_cell: 1.3993553601551832e-08
Optimal WT_surf_cell: 7.225636425853707e-09
Original WT_k_ILM_b: 0.04338001616481063
Optimal WT_k_ILM_b: 0.03570297267241735
Original WT_k_IL_RA_b: 0.00021989869945295703
Optimal WT_k_IL_RA_b: 0.0015148384362274662


In [29]:
# Update binding rates to the infered values and save them
df_bind = pd.read_csv('data/binding/IL10_data_param_ABC_SMC_IL10_RAp.csv')
variants_list = ["WT","Super-10","R5A11D"]
df_bind = update_df_bind_ABC(df_bind, param_dict['IL10_RAp'], param_depen, param_unique_var, variants_list)
df_bind.to_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE_WT.csv", index = False) # Save infered parameter values

In [30]:
history = pyabc.History("sqlite:///results/ABC_SMC/pyABC_inference_ODE_mut.db")
param_dict= {
    "IL10_RAp": {}
    }
model_i = 0
model = model_list[1]
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
print("Prior/posterior distr. plots for: "+model)
fin_pop = [particle for particle in history.get_population(t=history.max_t).get_particles_by_model()[model_i]]
variants_list = ["WT","Super-10","R5A11D"]
for variant in variants_list:
    df_fit_data_ODE = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10_ODE_mut.csv') # Parameters to fit
    df_fit_data_ODE["Var__Param"] = df_fit_data_ODE["Variant"]+"__"+df_fit_data_ODE["Parameter"]
    df_fit_data_ODE = df_fit_data_ODE[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data_ODE = df_fit_data_ODE.loc[(df_fit_data_ODE["Var__Param"].str.contains(variant))&(df_fit_data_ODE["Model"]==model)]
    
    df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10.csv') # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[(df_fit_data["Var__Param"].str.contains(variant))&(df_fit_data["Model"]==model)]
        
    # For each infered parameter in the variant
    for var__param in df_fit_data_ODE["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_vec_plots_dict[model][param]
    
        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        x = 10**x
        
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Get data from final time point
        df, w = history.get_distribution(m=model_i, t=history.max_t)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        
        # Change the binding rates dataset to include the fitted parameters (take the individual with lowest distance
        i = 0
        error_saved = 1
        i_saved = 0
        for particle in fin_pop:
            if particle.distance < error_saved:
                error_saved = particle.distance
                i_saved = i
            i += 1
        
        param_dict[model][var__param] = fin_pop[i_saved].parameter[var__param]
        
        # Get the evolution of the optimal parameter values with each iteration of the posterior
        for i in range(0,history.max_t+1):
            df, w = history.get_distribution(m=model_i, t=i)
            x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
            opt_param_evol.append(10**x[np.argmax(pdf)]) # Save value where prior is maximum for temporal evolution  
        
        # Save distribution channging from unbinding rate to Kd
        x = 10**x
                
        print("Optimal "+var+"_"+param+": " + str(10**fin_pop[i_saved].parameter[var__param]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")
    
        # Finalize plot with prior and posterior distributions
        plt.ylabel("Density", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(param_plot, fontsize=25)
        plt.yscale("log")
        plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
        middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
        plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots+'IL10_RAp_ODE/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.show()

Prior/posterior distr. plots for: IL10_RAp
Original WT_k_IL_RA_RB_b: 1.2494244287099834
Optimal WT_k_IL_RA_RB_b: 0.4068489819074392
Original WT_k_DEPHOS_R: 0.9995395429679866
Optimal WT_k_DEPHOS_R: 1.081065205490448
Original Super-10_k_IL_RA_RB_b: 0.013293875921474222
Optimal Super-10_k_IL_RA_RB_b: 0.015967032047947593
Original R5A11D_k_IL_RA_RB_b: 0.039981581718719475
Optimal R5A11D_k_IL_RA_RB_b: 0.08020713478105117


In [31]:
# Update binding rates to the infered values and save them
df_bind = pd.read_csv('data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE_WT.csv')
variants_list = ["WT","Super-10","R5A11D"]
df_bind = update_df_bind_ABC(df_bind, param_dict['IL10_RAp'], param_depen, param_unique_var, variants_list)
df_bind.to_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE.csv", index = False) # Save infered parameter values

## Retraining of parameters for the IL-10 signaling-capable monomeric variants 

In [32]:
history = pyabc.History("sqlite:///results/ABC_SMC/pyABC_inference_ODE_mut_IL10M.db")
param_dict= {
    "Control": {},
    "IL10_RAp": {}
    }
variants_list = ["WT","Super-10","R5A11D"]
model_i = 0
model = model_list[1]
df_bind = pd.read_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE.csv")
print("Prior/posterior distr. plots for: "+model)
fin_pop = [particle for particle in history.get_population(t=history.max_t-1).get_particles_by_model()[model_i]]
for variant in variants_list:
    df_fit_data = pd.read_csv(path_data+'/IL10/data/ABC_SMC/param_inference_IC_IL10M_ODE_mut.csv') # Parameters to fit
    df_fit_data["Var__Param"] = df_fit_data["Variant"]+"__"+df_fit_data["Parameter"]
    df_fit_data = df_fit_data[["Model","Var__Param","Initial_val","Lower_bound","Upper_bound"]]
    df_fit_data = df_fit_data.loc[(df_fit_data["Var__Param"].str.contains(variant))&(df_fit_data["Model"]==model)]
        
    # For each infered parameter in the variant
    for var__param in df_fit_data["Var__Param"]:
        var, param = var__param.split("__")
        param_plot = param_vec_plots_dict[model][param]

        # Plot prior and final posterior distributions
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        
        # Plot prior
        x = np.linspace(np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2,np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2,10000)
        pdf = pyabc.RV("norm", loc=np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]), scale=conf95_2SD/2).pdf(x)
        # If infered parameter is an unbinding rate in the plot we put the Kd
        x = 10**x
        
        opt_param_evol = [x[np.argmax(pdf)]] # Save value where prior is maximum for temporal evolution    
            
        # Plot prior
        plt.plot(x,pdf/np.sum(pdf), color="darkred", linewidth=7.5, label=r"$p(\theta)$")
        
        print("Original "+var+"_"+param+": " + str(x[np.argmax(pdf)]))
        
        # Get data from final time point
        df, w = history.get_distribution(m=model_i, t=history.max_t-1)
        x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
        
        # Change the binding rates dataset to include the fitted parameters (take the individual with lowest distance
        i = 0
        error_saved = 1
        i_saved = 0
        for particle in fin_pop:
            if particle.distance < error_saved:
                error_saved = particle.distance
                i_saved = i
            i += 1
        
        param_dict[model][var__param] = fin_pop[i_saved].parameter[var__param]
        
        # Get the evolution of the optimal parameter values with each iteration of the posterior
        for i in range(0,history.max_t-1+1):
            df, w = history.get_distribution(m=model_i, t=i)
            x, pdf = pyabc.visualization.kde.kde_1d(df, w, x=var__param,xmin = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])-2, xmax = np.log10(df_bind.loc[df_bind["Variant"]==var,param].values[0])+2, numx=10000)
            opt_param_evol.append(10**x[np.argmax(pdf)]) # Save value where prior is maximum for temporal evolution  
        
        # Save distribution channging from unbinding rate to Kd
        x = 10**x
                
        print("Optimal "+var+"_"+param+": " + str(10**fin_pop[i_saved].parameter[var__param]))    
        
        # Plot posterior
        plt.plot(x,pdf/np.sum(pdf), color="blue", linewidth=7.5, label=r"$p(\theta|D)$")

        # Finalize plot with prior and posterior distributions
        plt.ylabel("Density", fontsize=25)
        plt.xlabel(param_plot, fontsize=25)
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.legend(loc="best", fontsize=20)
        ax.set_yticks([])
        plt.xscale('log')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
        plt.savefig(directory_plots+'IL10M_RAp_ODE/Param_inference_distribution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
        # Plot evolution of optimal parameter values
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        plt.plot(range(-1,history.max_t-1+1),opt_param_evol, color="blue", linewidth=7.5)
        plt.xlabel("Iteration", fontsize=25)
        plt.ylabel(param_plot, fontsize=25)
        plt.yscale("log")
        plt.ylim(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Lower_bound"].values[0],df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Upper_bound"].values[0])
        middle_tick = 10**round(np.log10(df_fit_data.loc[df_fit_data["Var__Param"]==var__param,"Initial_val"].values[0]))
        plt.yticks([middle_tick/10, middle_tick, middle_tick*10])
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        plt.savefig(directory_plots +'IL10M_RAp_ODE/Param_inference_evolution_'+var+'__'+param+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()
        
    # Plot epsilon evolution
    x = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_xdata()
    y = pyabc.visualization.plot_epsilons(history).get_lines()[0].get_ydata()
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.plot(x,y, color="blue", linewidth=7.5)
    plt.xlabel("Iteration", fontsize=25)
    plt.ylabel(r'$\epsilon$', fontsize=25)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    plt.savefig(directory_plots +'IL10M_RAp_ODE/Param_inference_evolution_'+var+'__epsilon.pdf', transparent=True, bbox_inches="tight")
    plt.show()

Prior/posterior distr. plots for: IL10_RAp
Original WT_k_IL_RA_RB_b: 0.38764227468976464
Optimal WT_k_IL_RA_RB_b: 9.537167582717977
Original WT_k_DEPHOS_R: 1.2349735898179284
Optimal WT_k_DEPHOS_R: 3.699776757718795
Original R5A11D_k_IL_RA_RB_b: 0.049704100684687585
Optimal R5A11D_k_IL_RA_RB_b: 0.020694642589965558


In [33]:
# Update binding rates to the infered values and save them
df_bind = pd.read_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE.csv")
variants_list = ["WT","R5A11D","Super-10"]
df_bind = update_df_bind_ABC(df_bind, param_dict['IL10_RAp'], param_depen, param_unique_var, variants_list)
df_bind.to_csv("data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE_IL10M.csv", index = False) # Save infered parameter values